In [1]:
"""
class trial:

    def __init__(
        self
    ):  # other object variables that maybe needed can be added after the comma
        pass  # placeholder
"""
count = 0

def area(l,b):
    global count
    l+=count
    b+=count
    count+=1
    return l*b
    
def calc():
        
    for i in range(3):
        a = area(1,1)
        print (a)
    global count 
    count = 0

In [3]:
calc()

1
4
9


In [2]:
from scipy.spatial import ConvexHull
import numpy as np
from random_matrix.utils import integration_utils as iu

count = 0

def triangle(n):
    row = 100
    col = 101
    num_linear = row * col
    theta = np.linspace(0, np.pi, col)
    phi = np.linspace(0, 2 * np.pi, row)
    theta_grid, phi_grid = np.meshgrid(theta, phi)
    r = 1
    # Calculate x, y, z coordinates for the sphere
    x = np.reshape((r * np.sin(theta_grid) * np.cos(phi_grid)), (1, num_linear))
    y = np.reshape((r * np.sin(theta_grid) * np.sin(phi_grid)), (1, num_linear))
    z = np.reshape((r * np.cos(theta_grid)), (1, num_linear))
    point = np.transpose(np.vstack((x, y, z)))
    # Compute the convex hull
    hull = ConvexHull(point)
    return len(hull.simplices),point[hull.simplices[n]]
    

def GramMatrix(n):
    # Here n is the simplex number. Function returns the det of the Gram matrix for the nth simplex
    points = triangle(n)[1]
    v1 = points[1] - points[0]
    v2 = points[2] - points[0]
    J = np.array([[np.dot(v1, v1), np.dot(v1, v2)], [np.dot(v2, v1), np.dot(v2, v2)]])
    G = np.sqrt(np.linalg.det(J))
    return G


def cart2sph(x, y, z):  # NB convention on t differs from MATLAB
    xy = x**2 + y**2
    r = np.sqrt(xy + z**2)
    t = np.pi / 2 - np.arctan2(
        z, np.sqrt(xy)
    )  # for elevation angle defined from Z-axis down
    p = np.arctan2(y, x)
    if np.all(p<0):
        p=p+2*np.pi

    return (r,t,p)



def function(a, b):
    global count
    points = triangle(count)[1]
    vertex = points[0]
    v1 = points[1] - points[0]
    v2 = points[2] - points[0]
    x = vertex[0] + a * v1[0] + b * v2[0]
    y = vertex[1] + a * v1[1] + b * v2[1]
    z = vertex[2] + a * v1[2] + b * v2[2]
    r, theta, phi = cart2sph(x,y,z)
    #return r*theta*phi
    return GramMatrix(count)*phi

def integration():
    global count
    count = 0
    res = 0
    num = triangle(0)[0]
    for i in range(0,num):
        res = res + iu.basic_triangle_integral(function, [[0, 0], [1 ,0], [0, 1]])
        count+=1
    return res

    
    
     

In [3]:
from scipy.integrate import quad, dblquad
def integrand3(theta, phi):
    return theta*np.sin(theta)

# Compute the integral over [-1, 1]
result, _ = dblquad(integrand3, 0, 2 * np.pi, 0, np.pi)
print(result)

19.739208802178716


In [87]:
integration()

np.float64(39.06174648359799)